# conv-output-shape — ex1: compute conv2d output shape analytically

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-output-shape`. Running the final beacon cell reports progress against the `CNN: Conv output shape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv output shape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-output-shape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-output-shape"
DD_SUBTOPIC = "CNN: Conv output shape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv output shape — the formula

For a 2-D convolution with input `(B, IC, H, W)`, kernel `(OC, IC, KH, KW)`, **stride** `(SH, SW)`, **padding** `(PH, PW)`, and **dilation** `(DH, DW)`:

```
H_out = floor( (H + 2*PH - DH*(KH - 1) - 1) / SH ) + 1
W_out = floor( (W + 2*PW - DW*(KW - 1) - 1) / SW ) + 1
```

For the common case `dilation=1`, this simplifies to

```
H_out = floor( (H + 2*PH - KH) / SH ) + 1
W_out = floor( (W + 2*PW - KW) / SW ) + 1
```

**The shape rule.** Output is `(B, OC, H_out, W_out)` — the batch and kernel-output-channels axes pass through unchanged; `IC` is *contracted away* by the convolution; `H` and `W` shrink per the formula.

**Special cases worth memorizing:**
- `padding = (KH-1)//2` with stride 1, odd KH → `H_out = H` ("same" padding).
- `padding = 0`, stride 1 → `H_out = H - KH + 1` (the minimal no-pad form ARENA's `conv1d_minimal` uses).
- `stride = KH`, padding 0 → `H_out = H // KH` (non-overlapping tiles).

### Exercise 1 — compute conv2d output shape analytically

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the 2-D convolution output-shape formula `H_out = floor((H + 2P - K) / S) + 1` (and analogously for `W`) to predict the output shape `(B, OC, H_out, W_out)` from input shape, kernel size, stride, and padding.
> Keywords: conv, shape, formula, analytical
> ```

**KCs targeted:** `conv-output-shape-formula`, `conv-shape-batch-pass-through`

Implement `ex1_conv2d_outshape(input_shape, out_channels, kernel_size, stride, padding)`.

- `input_shape` is a tuple `(B, IC, H, W)`.
- `kernel_size`, `stride`, `padding` are each a 2-tuple `(h_val, w_val)`.
- Return a tuple `(B, out_channels, H_out, W_out)`.

**Formula (dilation=1).**
```
H_out = (H + 2*PH - KH) // SH + 1
W_out = (W + 2*PW - KW) // SW + 1
```

**Hint.** Use Python integer arithmetic — no tensors needed. The batch axis and `out_channels` axis pass through unchanged; the input channels axis `IC` is *contracted away* (does not appear in the output shape).

After your computation, the test creates an empty `nn.Conv2d` with the same hyperparams, runs the same input through it, and confirms your predicted shape matches the actual tensor's shape.

In [ ]:
def ex1_conv2d_outshape(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)


<details><summary>Solution</summary>

```python
def ex1_conv2d_outshape(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)
```

**Floor division is essential.** `(H + 2*PH - KH) / SH` may be fractional — convolution drops any partial window at the right edge. Use `//` (integer floor) to match PyTorch's behavior.

**Why `+ 1`.** With stride `S` and effective input length `L = H + 2*PH`, the number of valid kernel positions is `floor((L - K) / S) + 1`. The `+1` counts the *first* position (window starting at 0); the floored quotient counts the additional positions reachable by stepping `S` at a time.

**Pass-through axes.** `B` (batch) and `OC` (output channels) appear unchanged in the output. `IC` (input channels) **does not appear** — it's contracted by the dot product with the kernel's `IC` axis.

**Dilation generalization.** Replace `KH` with the *effective* kernel size `DH * (KH - 1) + 1` for dilation `DH > 1`. The drill fixes dilation=1 to keep the formula clean.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()